# Setup

In [ ]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import pandas as pd

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.io_utils import save_csv
from src.evidence import create_alert_evidence_package

alerts_df = pd.read_csv(PROJECT_ROOT / "outputs/alerts/alerts.csv")
global_df = pd.read_csv(PROJECT_ROOT / "outputs/global_ids/global_ids.csv")
rep_crops_df = pd.read_csv(PROJECT_ROOT / "outputs/crops/representative_crops.csv")

alerts_df.head()


# Tạo evidence package cho alert

In [ ]:
# Chuẩn bị ảnh và dữ liệu evidence để con người kiểm tra các cảnh báo quan trọng.
evidence_root = PROJECT_ROOT / "outputs/evidence"
evidence_root.mkdir(parents=True, exist_ok=True)

high_alerts = alerts_df[alerts_df["status"] == "alert_generated"].copy()

evidence_rows = []

for _, alert in high_alerts.iterrows():
    gid = alert["global_id"]

    gid_info = global_df[global_df["global_id"] == gid]

    if len(gid_info) == 0:
        continue

    entry_tracklet_id = int(gid_info.iloc[0]["entry_tracklet_id"])

    crop_candidates = rep_crops_df[
        rep_crops_df["local_track_id"] == entry_tracklet_id
    ].sort_values("quality_score", ascending=False)

    crop_paths = crop_candidates["crop_path"].head(3).tolist()

    gid_dir = create_alert_evidence_package(
        global_id=gid,
        alert_row=alert,
        crop_paths=crop_paths,
        evidence_root=evidence_root
    )

    evidence_rows.append({
        "global_id": gid,
        "evidence_folder": str(gid_dir),
        "num_crops": len(crop_paths)
    })

evidence_df = pd.DataFrame(evidence_rows)

out_path = PROJECT_ROOT / "outputs/evidence/evidence_index.csv"
save_csv(evidence_df, out_path)

print("Saved:", out_path)
evidence_df.head()


# Tạo human review CSV

In [ ]:
# Chuẩn bị ảnh và dữ liệu evidence để con người kiểm tra các cảnh báo quan trọng.
human_review_rows = []

for _, alert in high_alerts.iterrows():
    gid = alert["global_id"]

    evidence_folder = str(evidence_root / gid)

    human_review_rows.append({
        "global_id": gid,
        "suspicion_score": float(alert["suspicion_score"]),
        "alert_level": alert["alert_level"],
        "system_reason": alert["reason"],
        "evidence_folder": evidence_folder,
        "human_decision": "pending",
        "human_note": ""
    })

human_review_df = pd.DataFrame(human_review_rows)

out_path = PROJECT_ROOT / "outputs/human_review.csv"
save_csv(human_review_df, out_path)

print("Saved:", out_path)
human_review_df.head()
